# SNPs generation

In [5]:
import os
import glob
import numpy as np
import pandas as pd
from cyvcf2 import VCF

In [ ]:
path = "/Users/sijianfan/Documents/projects/BiSSGL/datasets/realAnalysis/tuberculosis"

WHO_FILE = f"{path}/who_mutations.csv"  # WHO 突变定义文件
VCF_PATTERN = f"{path}/assemblies/*.raw.vcf.gz"  # 匹配你的 VCF 文件名
OUTPUT = f"{path}/mutation_matrix_who_sites.csv"
MIN_COV = 10  # 覆盖度阈值（低于该值视为不可调用 NaN）

In [7]:
print("Loading WHO mutation list...")
who = pd.read_csv(WHO_FILE, dtype={"pos": int})
required = {"mut_name", "pos", "ref", "alt"}
if not required.issubset(who.columns):
    raise ValueError(f"who_mutations.csv must contain columns: {required}")

# 去重（安全）
who = who.drop_duplicates(subset=["mut_name"])
mut_names = who["mut_name"].tolist()
print(f"Loaded {len(mut_names)} WHO mutations.")

Loading WHO mutation list...
Loaded 118431 WHO mutations.


In [8]:
# 查找所有 VCF
vcf_files = sorted(glob.glob(VCF_PATTERN))
if len(vcf_files) == 0:
    raise RuntimeError(f"No VCF files found matching: {VCF_PATTERN}")

print(f"Found {len(vcf_files)} VCF files.")

Found 7932 VCF files.


In [9]:
rows = []
index = []
# -------------------- 主循环：对每个 assembly 处理一次 --------------------
for vcf_path in vcf_files:
    acc = os.path.basename(vcf_path).replace(".raw.vcf.gz", "")
    print(f"Processing {acc} ...")

    # 单个 VCF 的突变结果（初始为 NaN）
    row = {m: np.nan for m in mut_names}

    v = VCF(vcf_path)

    # 建立位置索引：pos → list(records)
    pos_dict = {}
    for rec in v:
        pos_dict.setdefault(int(rec.POS), []).append(rec)

    # 针对每个 WHO 突变
    for _, w in who.iterrows():
        pos = int(w["pos"])
        ref_w = str(w["ref"])
        alt_w = str(w["alt"])
        mut = w["mut_name"]

        recs = pos_dict.get(pos, [])
        if len(recs) == 0:
            # 没有记录 → 无法调用
            row[mut] = np.nan
            continue

        seen = False
        callable_flag = False

        for rec in recs:
            # 取覆盖度 DP（只有一个 sample）
            try:
                dp_arr = rec.format("DP")
                dp = int(dp_arr[0]) if dp_arr is not None else 0
            except:
                dp = 0

            if dp < MIN_COV:
                continue  # coverage 不足

            callable_flag = True
            # REF / ALT 匹配
            rec_ref = rec.REF
            rec_alts = [str(a) for a in rec.ALT]

            if rec_ref == ref_w and alt_w in rec_alts:
                seen = True

        if not callable_flag:
            row[mut] = np.nan
        else:
            row[mut] = 1 if seen else 0

    rows.append(row)
    index.append(acc)

Processing GCA_000008585.1 ...
Processing GCA_000016145.1 ...
Processing GCA_000016925.1 ...
Processing GCA_000023625.1 ...
Processing GCA_000154605.2 ...
Processing GCA_000155185.1 ...
Processing GCA_000159735.1 ...
Processing GCA_000159755.1 ...
Processing GCA_000162995.1 ...
Processing GCA_000163015.1 ...
Processing GCA_000176315.1 ...
Processing GCA_000184045.1 ...
Processing GCA_000195955.2 ...
Processing GCA_000220415.1 ...
Processing GCA_000220435.1 ...
Processing GCA_000220455.1 ...
Processing GCA_000270365.1 ...
Processing GCA_000296155.1 ...
Processing GCA_000296175.1 ...
Processing GCA_000313215.2 ...
Processing GCA_000364945.1 ...
Processing GCA_000372545.2 ...
Processing GCA_000411895.1 ...
Processing GCA_000442795.1 ...
Processing GCA_000442945.1 ...
Processing GCA_000442965.1 ...
Processing GCA_000443005.1 ...
Processing GCA_000443305.1 ...
Processing GCA_000443325.1 ...
Processing GCA_000443345.1 ...
Processing GCA_000443365.1 ...
Processing GCA_000443405.1 ...
Processi

KeyboardInterrupt: 

In [28]:
# -------------------- 保存为矩阵 --------------------
mat = pd.DataFrame(rows, index=index)
mat.index.name = "assembly_acc"
mat = mat.reindex(columns=mut_names)

In [31]:
mat.sum(axis=1)

assembly_acc
GCA_000008585.1    0.0
GCA_000016145.1    0.0
GCA_000016925.1    0.0
GCA_000023625.1    0.0
GCA_000154605.2    0.0
GCA_000155185.1    0.0
GCA_000159735.1    0.0
dtype: float64

In [ ]:
print(f"Writing: {OUTPUT}  shape={mat.shape}")
mat.to_csv(OUTPUT)
print("Done.")